# 04 - Demand Forecasting
## Time-Series Forecasting with Prophet

This notebook builds demand forecasting models to predict future order volume
and revenue by product category.

### Output
- `trained_models/forecaster_models.joblib` — Prophet models per category

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print('Prophet loaded successfully!')
except ImportError:
    PROPHET_AVAILABLE = False
    print('Prophet not installed. Install with: pip install prophet')
    print('Falling back to simple moving average forecasting.')

sys.path.insert(0, str(Path('.').resolve().parent))
from data.mapping import BRL_TO_DZD, CATEGORY_TRANSLATION

sns.set_style('whitegrid')
MODEL_DIR = Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded!')

## 1. Load & Prepare Data

In [ ]:
DATA_DIR = Path('../data/olist')

orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')

# Merge
df = orders[orders['order_status'] == 'delivered'].copy()
df['order_date'] = pd.to_datetime(df['order_purchase_timestamp'])

pay_agg = payments.groupby('order_id').agg(payment_value=('payment_value', 'sum')).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')

items_cat = items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
cat_per_order = items_cat.groupby('order_id')['product_category_name'].first().reset_index()
df = df.merge(cat_per_order, on='order_id', how='left')

df['revenue_dzd'] = (df['payment_value'].fillna(0) * BRL_TO_DZD).round(2)
df['category_en'] = df['product_category_name'].map(
    lambda c: CATEGORY_TRANSLATION.get(str(c), str(c)) if pd.notna(c) else 'unknown'
)

print(f'Delivered orders: {len(df)}')
print(f'Date range: {df["order_date"].min().date()} to {df["order_date"].max().date()}')

## 2. Prepare Time Series

In [ ]:
# Daily revenue time series
daily = df.groupby(df['order_date'].dt.date).agg(
    y=('revenue_dzd', 'sum'),
    order_count=('order_id', 'count'),
).reset_index()
daily.columns = ['ds', 'y', 'order_count']
daily['ds'] = pd.to_datetime(daily['ds'])

# Fill missing days
full_range = pd.date_range(daily['ds'].min(), daily['ds'].max(), freq='D')
daily = daily.set_index('ds').reindex(full_range, fill_value=0).reset_index()
daily.columns = ['ds', 'y', 'order_count']

print(f'Time series: {len(daily)} days')

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(daily['ds'], daily['y'], color='steelblue', alpha=0.7)
axes[0].set_title('Daily Revenue (DZD)')
axes[0].set_ylabel('Revenue')

# 7-day moving average
daily['y_ma7'] = daily['y'].rolling(7).mean()
axes[0].plot(daily['ds'], daily['y_ma7'], color='red', linewidth=2, label='7-day MA')
axes[0].legend()

axes[1].bar(daily['ds'], daily['order_count'], color='orange', alpha=0.7, width=1)
axes[1].set_title('Daily Order Count')
axes[1].set_ylabel('Orders')

plt.tight_layout()
plt.show()

## 3. Train Prophet Models

In [ ]:
models = {}

if PROPHET_AVAILABLE:
    # Train overall model
    print('Training overall revenue model...')
    prophet_all = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
    )
    prophet_all.fit(daily[['ds', 'y']])
    models['all'] = prophet_all
    print('  Overall model trained!')
    
    # Train top-5 category models
    top_categories = df['category_en'].value_counts().head(5).index.tolist()
    
    for cat in top_categories:
        print(f'Training model for: {cat}...')
        cat_df = df[df['category_en'] == cat]
        cat_daily = cat_df.groupby(cat_df['order_date'].dt.date).agg(
            y=('revenue_dzd', 'sum')
        ).reset_index()
        cat_daily.columns = ['ds', 'y']
        cat_daily['ds'] = pd.to_datetime(cat_daily['ds'])
        
        # Fill missing days
        cat_daily = cat_daily.set_index('ds').reindex(full_range, fill_value=0).reset_index()
        cat_daily.columns = ['ds', 'y']
        
        prophet_cat = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
        )
        prophet_cat.fit(cat_daily)
        models[cat] = prophet_cat
        print(f'  {cat} model trained!')
    
    print(f'\nTotal models trained: {len(models)}')
else:
    print('Prophet not available. Saving metadata for simple forecasting fallback.')
    # Store daily stats for the simple forecaster
    models['all'] = {'daily_data': daily[['ds', 'y']].to_dict()}
    print('Metadata saved for fallback forecasting.')

## 4. Generate & Visualize Forecasts

In [ ]:
if PROPHET_AVAILABLE:
    # Forecast 30 days
    future = models['all'].make_future_dataframe(periods=30)
    forecast = models['all'].predict(future)
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # Overall forecast
    axes[0].plot(daily['ds'], daily['y'], 'b.', alpha=0.3, label='Actual')
    axes[0].plot(forecast['ds'], forecast['yhat'], 'r-', label='Forecast')
    axes[0].fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                         alpha=0.2, color='red', label='Confidence Interval')
    axes[0].set_title('30-Day Revenue Forecast (DZD)')
    axes[0].legend()
    
    # Components
    models['all'].plot_components(forecast, ax=None)
    plt.tight_layout()
    plt.show()
    
    # Print predictions
    future_only = forecast.tail(30)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    future_only['yhat'] = future_only['yhat'].clip(lower=0).round(0)
    future_only['yhat_lower'] = future_only['yhat_lower'].clip(lower=0).round(0)
    future_only['yhat_upper'] = future_only['yhat_upper'].clip(lower=0).round(0)
    print('\n30-Day Forecast (Daily Revenue in DZD):')
    print(future_only.to_string(index=False))
else:
    print('Prophet not available - skipping forecast visualization')
    ma7 = daily['y'].tail(7).mean()
    ma30 = daily['y'].tail(30).mean()
    print(f'\nSimple Forecast (Moving Average):')
    print(f'  7-day MA: {ma7:,.0f} DZD/day')
    print(f'  30-day MA: {ma30:,.0f} DZD/day')
    print(f'  Projected 30-day revenue: {ma7 * 30:,.0f} DZD')

## 5. Save Models

In [ ]:
joblib.dump(models, MODEL_DIR / 'forecaster_models.joblib')

print(f'Forecasting models saved to {MODEL_DIR / "forecaster_models.joblib"}')
print(f'Models for categories: {list(models.keys())}')
print('\nDone! The forecasting models are ready for the FastAPI service.')